# Model Comparison with MLflow Experiment Tracking

Compares all five forecasting approaches on the same 60-day held-out test set (2015-06-02 to 2015-07-31).

Each model is logged as a separate MLflow run inside the `rossmann-demand-forecasting` experiment so metrics can be compared side-by-side in the MLflow UI.

Run `mlflow ui --backend-store-uri sqlite:///mlflow.db` from the project root, then open http://localhost:5000.

**Models compared:**
- Naive
- Seasonal Naive
- ARIMA(7,0,0)
- SARIMA(1,0,0)x(1,0,1,7)
- LightGBM

## 1. Imports

In [ ]:
import os

import mlflow
import pandas as pd
import matplotlib.pyplot as plt

# SQLite backend -- zero setup, fully local, works with MLflow 2.x and 3.x
# mlflow.db is created automatically in the project root on first run
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("rossmann-demand-forecasting")

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print("Active experiment   : rossmann-demand-forecasting")

## 2. Model Results

Metrics come from their respective training notebooks:
- Classical models: `02_forecasting_classical.ipynb`
- LightGBM: `03_forecasting_modern.ipynb`

All models evaluated on the same 60-day holdout (2015-06-02 to 2015-07-31).

In [ ]:
model_configs = [
    {
        "name": "Naive",
        "type": "classical",
        "params": {"method": "last_known_value"},
        "mae": 2746028.78,
        "rmse": 4278452.33,
    },
    {
        "name": "Seasonal Naive",
        "type": "classical",
        "params": {"method": "same_day_last_week", "seasonality": 7},
        "mae": 2084717.68,
        "rmse": 2570396.03,
    },
    {
        "name": "ARIMA",
        "type": "classical",
        "params": {"order": "(7,0,0)", "p": 7, "d": 0, "q": 0},
        "mae": 2075114.51,
        "rmse": 2727431.88,
    },
    {
        "name": "SARIMA",
        "type": "classical",
        "params": {
            "order": "(1,0,0)x(1,0,1,7)",
            "p": 1, "d": 0, "q": 0,
            "P": 1, "D": 0, "Q": 1, "s": 7,
        },
        "mae": 1122513.00,
        "rmse": 1540606.00,
    },
    {
        "name": "LightGBM",
        "type": "ml",
        "params": {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "num_leaves": 31,
            "random_state": 42,
            "features": (
                "DayOfWeek,Promo,SchoolHoliday,StateHoliday_Count,"
                "DayOfWeek_Num,Month,Year,DayOfMonth,"
                "Sales_Lag_1,Sales_Lag_7,Sales_Lag_14,"
                "Sales_Rolling_Mean_7,Sales_Rolling_Mean_14"
            ),
            "train_end": "2015-06-01",
            "scope": "chain_wide_daily_aggregate",
        },
        "mae": 323834.04,
        "rmse": 451188.96,
    },
]

model_results = pd.DataFrame([
    {"Model": m["name"], "MAE": m["mae"], "RMSE": m["rmse"]}
    for m in model_configs
])
model_results

## 3. Log Each Model as an MLflow Run

In [ ]:
run_ids = {}

for cfg in model_configs:
    with mlflow.start_run(run_name=cfg["name"]) as run:
        mlflow.set_tag("model_type", cfg["type"])
        mlflow.set_tag("scope", "chain_wide")
        mlflow.set_tag("test_period", "2015-06-02_to_2015-07-31")
        mlflow.set_tag("notebook", "04_model_comparison")

        for k, v in cfg["params"].items():
            mlflow.log_param(k, v)

        mlflow.log_metric("mae", cfg["mae"])
        mlflow.log_metric("rmse", cfg["rmse"])

        run_ids[cfg["name"]] = run.info.run_id
        print(f"Logged: {cfg['name']:30s}  MAE={cfg['mae']:>12,.0f}  run_id={run.info.run_id}")

print("\nAll models logged.")

## 4. Log Comparison Charts as Artifacts

In [ ]:
os.makedirs("../figures/04_model_comparison", exist_ok=True)

for metric, label, fname in [
    ("MAE",  "Mean Absolute Error",    "model_mae_comparison.png"),
    ("RMSE", "Root Mean Squared Error", "model_rmse_comparison.png"),
]:
    df = model_results.sort_values(metric)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(df["Model"], df[metric])
    ax.set_title(f"Model Comparison: {label} (Lower is Better)")
    ax.set_xlabel("Model")
    ax.set_ylabel(metric)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    path = f"../figures/04_model_comparison/{fname}"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()

# Attach both charts to the LightGBM run
with mlflow.start_run(run_id=run_ids["LightGBM"]):
    mlflow.log_artifact("../figures/04_model_comparison/model_mae_comparison.png",  "comparison_charts")
    mlflow.log_artifact("../figures/04_model_comparison/model_rmse_comparison.png", "comparison_charts")
print("Charts attached to LightGBM run.")

## 5. Rank Models and Log Improvement Metrics

In [ ]:
comparison_ranked = model_results.copy()
comparison_ranked["MAE Rank"]  = comparison_ranked["MAE"].rank(method="min").astype(int)
comparison_ranked["RMSE Rank"] = comparison_ranked["RMSE"].rank(method="min").astype(int)
comparison_ranked = comparison_ranked.sort_values("MAE").reset_index(drop=True)
print(comparison_ranked.to_string(index=False))

sarima_mae  = model_results.loc[model_results["Model"] == "SARIMA", "MAE"].iloc[0]
sarima_rmse = model_results.loc[model_results["Model"] == "SARIMA", "RMSE"].iloc[0]
lgbm_mae    = model_results.loc[model_results["Model"] == "LightGBM", "MAE"].iloc[0]
lgbm_rmse   = model_results.loc[model_results["Model"] == "LightGBM", "RMSE"].iloc[0]

mae_improvement  = (sarima_mae  - lgbm_mae)  / sarima_mae  * 100
rmse_improvement = (sarima_rmse - lgbm_rmse) / sarima_rmse * 100

print(f"\nLightGBM vs SARIMA -- MAE  reduction: {mae_improvement:.2f}%")
print(f"LightGBM vs SARIMA -- RMSE reduction: {rmse_improvement:.2f}%")

with mlflow.start_run(run_id=run_ids["LightGBM"]):
    mlflow.log_metric("mae_improvement_vs_sarima_pct",  round(mae_improvement,  2))
    mlflow.log_metric("rmse_improvement_vs_sarima_pct", round(rmse_improvement, 2))
    mlflow.set_tag("deployed", "true")
    mlflow.set_tag("endpoint", "/forecast")

print("Improvement metrics + deployed tag logged to LightGBM run.")

## 6. Run IDs Summary

In [ ]:
print("Experiment: rossmann-demand-forecasting")
print("-" * 65)
for name, run_id in run_ids.items():
    print(f"  {name:30s}  {run_id}")

print("\nTo view in UI (run from project root):")
print("  mlflow ui --backend-store-uri sqlite:///mlflow.db")
print("  then open http://localhost:5000")